# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata

print(f"Dataset Name: {md.name}\nDescription: {md.description}")

## 2. Data Overview
Review available record sets, their fields, columns, and their `@id`s.

Below, we enumerate all available record sets, their field `@id`s, and provide example data for each. This will help you identify the correct `@id` values to use when referencing entities in further analyses.


In [ ]:
# List all record sets with their @id's and details
print('Available Record Sets:')
for rs in md.record_sets:
    print(f"\nRecordSet name: {rs.name}, @id: {rs.id_}")
    print('  Fields:')
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id_}, type: {f.data_type})")
    print('  Columns:')
    for col in getattr(rs, 'columns', []):
        print(f"    - {col.name} (@id: {col.id_}, source: {col.source})")


## Preview First Rows of Each Record Set
Let's look at the first few records from each available record set using their `@id`.


In [ ]:
# Iterate record sets and print a preview of first few records
for rs in md.record_sets:
    print(f"---\nRecordSet: {rs.name} (@id: {rs.id_})")
    recs = list(dataset.records(record_set=rs.id_))
    print(f"Sample record (first row):\n{recs[0] if recs else 'No records.'}\n")

## 3. Data Extraction
Load data from specific record sets into Pandas DataFrames for analysis. All record set and field references are specified using their `@id` for consistency and reproducibility.


In [ ]:
# Build list of record set @id's
record_set_ids = [rs.id_ for rs in md.record_sets]
print(f"Record Sets to load: {record_set_ids}")

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for RecordSet {rs_id}")

# Let's print the columns of the first record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nColumns in primary record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Now, we will apply common data processing steps, such as filtering, normalization, and grouping, referencing all entities by their `@id`. If the main record set contains numeric fields, we demonstrate EDA on one of them.


In [ ]:
# Pick a numeric field from the primary record set by @id
# We'll choose the first numeric (float or integer) field available
numeric_field_id = None
group_field_id = None
rs = next((r for r in md.record_sets if r.id_ == main_rs_id), None)
for f in (rs.fields if rs else []):
    if f.data_type in ('Integer', 'Float', 'Number') and not numeric_field_id:
        numeric_field_id = f.id_
    if not group_field_id and f.data_type == 'Text':
        group_field_id = f.id_

print(f"Numeric field selected for EDA: {numeric_field_id}")
if group_field_id:
    print(f"Will use {group_field_id} for grouping.")

df = dataframes[main_rs_id]
if numeric_field_id and numeric_field_id in df.columns:
    numeric_col = df[numeric_field_id]

    # Coerce to numeric in case field is read as string
    numeric_col = pd.to_numeric(numeric_col, errors='coerce')
    threshold = numeric_col.quantile(0.75)
    filtered_df = df[numeric_col > threshold].copy()

    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (75th percentile):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_col[numeric_col > threshold] - numeric_col.mean()) / numeric_col.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a text/category field if available
    if group_field_id and group_field_id in df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped means by {group_field_id}:")
        display(grouped.head())
else:
    print("No numeric field found for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All variables are referenced by their `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to explore a dataset defined by the Croissant schema, referencing all entities by their `@id`. We loaded the metadata, listed and explored all record sets, extracted tables, performed basic EDA including normalization and grouping, and visualized data distributions. This approach ensures reproducibility and clarity for users working with FAIR datasets.
